# Fink/LSST — AGN Light Curves in the Deep Drilling Fields

This notebook searches for **Active Galactic Nuclei (AGN)** detected in the **LSST Deep
Drilling Fields (DDFs)** using the **Fink broker** alert stream, and downloads their
full photometric light curves (diaSources **and** forced photometry).

## Strategy

1. **Cone-search** each DDF via `api/v1/conesearch` (Fink/LSST API, `r:`/`f:` prefixes)
2. **Deduplicate** by `diaObjectId` and aggregate the Fink crossmatch columns (`f:xm_*`)
   per object (mode over all alerts — a given object can have inconsistent per-alert
   crossmatch values)
3. **Select AGN candidates** with the `classify_agn()` function (see below) combining:
   - SIMBAD/CDS object type (`f:xm_simbad_otype`) — confirmed AGN/QSO/Seyfert/blazar types
   - High-energy blazar catalogues (`f:xm_x3hsp_type`, `f:xm_x4lac_type`)
   - Fink CATS ML classifier (`f:clf_cats_class == 22` → *Non-periodic*, the CATS class
     that contains AGN-like stochastic variability, see Leoni et al. 2024,
     https://arxiv.org/abs/2404.08798)
   - Nearby-galaxy crossmatch (Mangrove / HyperLEDA) as a weak supporting indicator
     (AGN live in galaxy nuclei)
4. **Enrich** the catalogue with `/api/v1/objects` (all aggregated flux statistics,
   `firstDiaSourceMjdTai`, `lastDiaSourceMjdTai`, `nDiaSources`, etc.)
5. **Apply light-curve-quality cuts** tuned for AGN science: AGN variability is
   **stochastic / red-noise**, not periodic, and only becomes measurable over
   **long observational baselines** with **many epochs** — see the dedicated
   discussion in §6 below.
6. **Download full light curves** via `api/v1/sources` (`src`) + `api/v1/fp` (forced
   photometry, `fp`) for every selected AGN candidate, in every DDF.
7. **Plot** light curves (flux and magnitude), sky maps, and save everything to
   Parquet/CSV for downstream AGN variability analysis (structure functions,
   damped random walk fits, etc. in a follow-up notebook).

## Why AGN light curves?

Unlike periodic variables (Cepheids, RR Lyrae, eclipsing binaries) or single-epoch
transients (SNe), AGN show **continuous, aperiodic, low-amplitude optical variability**
driven by stochastic accretion-disc processes (often modelled as a **damped random
walk** / **Ornstein–Uhlenbeck process**, e.g. Kelly et al. 2009, MacLeod et al. 2010).
Characterising this requires:
- A **long time baseline** (ideally longer than the AGN's characteristic damping
  timescale, from tens to hundreds of days)
- **Densely sampled, well-calibrated multi-band photometry** — exactly what the
  LSST DDFs provide thanks to their high cadence
- Robust separation from **dipole/subtraction artefacts** (`r:isDipole`) and from
  **periodic variable stars** that can contaminate a naive selection

This notebook is the AGN-in-DDF analogue of `04_calib/01_fink_block_flatlightcurves.ipynb`
(stable-star calibration light curves) and `09_Cepheids/01_cepheids_in_DDF.ipynb`
(periodic variable light curves).

- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- created : 2026-07-03
- last update : 2026-07-03

## LSST Deep Drilling Fields used in this notebook

```
| Name of Field | RA (deg) | Dec (deg) | Type |
| -------------- | -------- | --------- | ---- |
| COSMOS         | 150.1191 | +2.2058   | DDF  |
| ELAIS-S1       | 9.4500   | -44.000   | DDF  |
| XMM-LSS        | 35.7080  | -4.750    | DDF  |
| ECDFS          | 53.1250  | -27.800   | DDF  |
| EDFS-a         | 58.9000  | -49.315   | DDF  |
| EDFS-b         | 63.6000  | -47.600   | DDF  |
| EDFS           | 61.2400  | -48.423   | DDF (combined a+b pointing) |
| M49            | 187.400  | +8.000    | Galaxy field |
```

## Fink/LSST API endpoints used (from `swagger.json`)

```
POST https://api.lsst.fink-portal.org/api/v1/conesearch   → alerts in a sky cone (r:/f: prefixes)
POST https://api.lsst.fink-portal.org/api/v1/sources      → full diaSources for one/many diaObjectId
POST https://api.lsst.fink-portal.org/api/v1/fp           → forced photometry for one/many diaObjectId
POST https://api.lsst.fink-portal.org/api/v1/objects      → aggregated object-level statistics (i:/r: prefix)
POST https://api.lsst.fink-portal.org/api/v1/blocks       → block flag definitions (informational)
POST https://api.lsst.fink-portal.org/api/v1/tags         → Fink classification tags (informational)
```

**Key API facts** (confirmed from live use in `04_calib` and `09_Cepheids`):
- `/api/v1/conesearch` requires the `r:` or `f:` column prefix (never `i:` — HTTP 500).
- `/api/v1/sources` and `/api/v1/fp` accept a single `diaObjectId` **or** a
  comma-separated list, and a `columns=` filter.
- `/api/v1/objects` accepts a comma-separated list of `diaObjectId` and returns **one
  row per object** with all aggregated per-band flux statistics — ideal to batch-fetch
  metadata for hundreds of candidates in a handful of calls.
- Cone-search radius is capped at 18000 arcsec (5 deg); we stay well below that.

## Relevant crossmatch / classifier columns for AGN selection

| Column | Catalogue | Notes |
|--------|-----------|-------|
| `f:xm_simbad_otype` | SIMBAD | object type string; AGN-like: `AGN`, `QSO`, `Sy1`, `Sy2`, `Sy`, `Bla`, `BLL`, `BlL`, `LIN`, `AGN_Candidate`, `QSO_Candidate` |
| `f:xm_x3hsp_type` | 3HSP | High-Synchrotron-Peaked blazar catalogue — blazars are AGN with a relativistic jet pointed at us |
| `f:xm_x4lac_type` | 4LAC | Fermi-LAT 4th AGN catalogue — gamma-ray-detected AGN/blazars |
| `f:xm_mangrove_2MASS_name` | Mangrove | 2MASS name of a nearby host galaxy (weak supporting evidence — AGN sit in galaxy nuclei) |
| `f:xm_mangrove_HyperLEDA_name` | Mangrove/HyperLEDA | HyperLEDA host-galaxy name |
| `f:xm_legacydr8_pstar` | Legacy DR8 | P(star); AGN hosts should have **low** P(star) (extended/galaxy-like) |
| `f:xm_legacydr8_zphot` | Legacy DR8 | photometric redshift of the (candidate) host galaxy |
| `f:clf_cats_class` | Fink ML (CATS) | broad-class classifier: `-1`=not processed, `11`=SN-like, `12`=Fast, `13`=Long, `21`=Periodic, **`22`=Non-periodic (e.g. AGN)**. See https://arxiv.org/abs/2404.08798 |
| `f:clf_cats_score` | Fink ML (CATS) | classifier probability [0-1] for the predicted class |
| `f:clf_snnSnVsOthers_score` | Fink ML | SN-vs-Others score — used to *reject* SN-like contaminants |
| `f:xm_tns_fullname` / `f:xm_tns_type` | TNS | occasionally AGN flares are typed `AGN` on TNS |
| `r:extendedness` | LSST | morphology: 0=point-source (unresolved nucleus dominates), 1=extended (resolved host galaxy) |
| `r:nDiaSources` | LSST | number of alerts/detections for the object — proxy for sampling density |
| `r:firstDiaSourceMjdTai`, `r:lastDiaSourceMjdTai` | LSST (via `/objects`) | first/last detection MJD — gives the **observational baseline**, the key AGN-selection quantity |
| `r:isDipole`, `r:dipole*` | LSST DIA | dipole/subtraction-artefact flags — used to *reject* spurious detections, not real AGN variability |

## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os, sys
import time
import collections
import warnings

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found -> interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found -> falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# to enlarge the sizes
params = {
    "legend.fontsize": "large",
    "axes.labelsize": "large",
    "axes.titlesize": "large",
    "xtick.labelsize": "large",
    "ytick.labelsize": "large",
}
plt.rcParams.update(params)

In [ ]:
# -- Fink API --------------------------------------------------------------
FINK_API = "https://api.lsst.fink-portal.org"

# -- Search parameters -------------------------------------------------------
NP_MIN = 30  # minimum nDiaSources to keep an object at the cone-search stage
CONE_RADIUS = 3600.0  # cone search radius in arcsec (1.0 deg per DDF)
N_ALERTS_MAX = 10000  # max alerts per cone-search call
SNR_MIN = 3.0  # minimum flux SNR for light curve points
BANDS = list("ugrizy")

# -- AGN light-curve-suitability cuts (see section 6 for the rationale) ------
# NP_MIN_AGN = 40  # minimum number of diaSources for a usable AGN light curve
NP_MIN_AGN = 500  # minimum number of diaSources for a usable AGN light curve
# MIN_BASELINE_DAYS = 120.0    # minimum observational baseline (first->last detection)
MIN_BASELINE_DAYS = 80.0
NC_PLOT = 30  # max number of light curves to plot per group

# Gaia parallax SNR threshold, reused only to help VETO stellar contaminants
GAIA_RPLX_MIN = 5.0

# -- Extended (tiled) cone-search fallback parameters -------------------------
# A plain conesearch is capped at N_ALERTS_MAX alerts. Dense fields (COSMOS in
# particular) saturate this cap, returning only a partial, epoch-biased subset
# of alerts -- which truncates nDiaSources and, critically, the observational
# baseline (firstDiaSourceMjdTai/lastDiaSourceMjdTai) used by the AGN cuts in
# section 6. When a field's plain conesearch is empty OR returns close to
# N_ALERTS_MAX alerts (a sign of truncation), we fall back to a spatially
# tiled search that sums many small-radius conesearches covering the same
# sky area, exactly as in 09_Cepheids/02_cepheids_extended_search.ipynb.
SATURATION_FRAC = 0.95  # trigger tiled fallback if len(df_cone) >= SATURATION_FRAC * N_ALERTS_MAX
TILE_RADIUS_MAX = 600.0  # arcsec, max tile radius for the fallback
TILE_RADIUS_MIN = 250.0  # arcsec, min tile radius for the fallback
FORCE_RELOAD_CONE = False  # set True to ignore the per-field parquet cache and re-fetch from the API

# -- LSST Deep Drilling Fields (RA/Dec J2000) --------------------------------
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# -- Output directories -------------------------------------------------------
NB_TAG = "AGN_DDF_01"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

# -- Plot style ----------------------------------------------------------------
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    """Save current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Fink API wrappers

In [ ]:
def _post_json(url: str, payload: dict, timeout: int = 120):
    """POST a JSON payload and return the parsed response, raising on HTTP errors
    (with the response body included in the exception message for easier debugging)."""
    r = requests.post(url, json=payload, timeout=timeout)
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        body = (r.text or "").strip()
        msg = f"{e}"
        if body:
            msg += f" | response: {body[:500]}"
        raise requests.HTTPError(msg, response=r) from e
    return r.json()


def fetch_conesearch(
    ra: float, dec: float, radius: float, n: int = N_ALERTS_MAX, columns=None
) -> pd.DataFrame:
    """
    Cone search via /api/v1/conesearch.
    IMPORTANT: column names must use the 'r:' or 'f:' prefix -- NOT 'i:' (HTTP 500).
    Returns one row per alert/diaSource (not per object).
    """
    payload = {"ra": ra, "dec": dec, "radius": radius, "n": n, "output-format": "json"}
    if columns:
        payload["columns"] = columns
    try:
        raw = _post_json(f"{FINK_API}/api/v1/conesearch", payload)
        if not raw:
            return pd.DataFrame()
        return pd.DataFrame(raw)
    except Exception as e:
        print(f"fetch_conesearch ERROR (ra={ra:.3f}, dec={dec:.3f}): {e}")
        return pd.DataFrame()


def fetch_sources(diaObjectId, columns=None) -> pd.DataFrame:
    """Fetch diaSources (direct detections) for one diaObjectId."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/sources", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_fp(diaObjectId, columns=None) -> pd.DataFrame:
    """Fetch forced photometry for one diaObjectId."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/fp", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_objects_batch(diaObjectIds: list, columns=None) -> pd.DataFrame:
    """
    Fetch aggregated object-level statistics for a list of diaObjectIds in a single
    API call (comma-separated IDs) via /api/v1/objects -- one row per diaObjectId.
    Much more efficient than one call per object.
    """
    ids_str = ",".join(str(oid) for oid in diaObjectIds)
    payload = {"diaObjectId": ids_str, "output-format": "json"}
    if columns:
        payload["columns"] = columns
    try:
        raw = _post_json(f"{FINK_API}/api/v1/objects", payload)
        return pd.DataFrame(raw) if raw else pd.DataFrame()
    except Exception as e:
        print(f"fetch_objects_batch ERROR for {len(diaObjectIds)} IDs: {e}")
        return pd.DataFrame()


def fetch_blocks() -> pd.DataFrame:
    """Fetch the Fink block definitions from /api/v1/blocks (informational only)."""
    url = f"{FINK_API}/api/v1/blocks"
    try:
        r = requests.get(url)
        if r.status_code in (404, 405):
            print(f"fetch_blocks: endpoint not available (HTTP {r.status_code})")
            return pd.DataFrame()
        r.raise_for_status()
        data = r.json()
        if not data:
            return pd.DataFrame()
        if isinstance(data, list):
            return pd.DataFrame(data) if isinstance(data[0], dict) else pd.DataFrame({"name": data})
        if isinstance(data, dict):
            return pd.DataFrame([data])
        return pd.DataFrame({"raw": [data]})
    except Exception as e:
        print(f"fetch_blocks unexpected error: {type(e).__name__}: {e}")
        return pd.DataFrame()


def fetch_tags() -> pd.DataFrame:
    """Fetch the Fink classification tags from /api/v1/tags (informational only)."""
    url = f"{FINK_API}/api/v1/tags"
    try:
        r = requests.get(url)
        if r.status_code in (404, 405):
            print(f"fetch_tags: endpoint not available (HTTP {r.status_code})")
            return pd.DataFrame()
        r.raise_for_status()
        data = r.json()
        if not data:
            return pd.DataFrame()
        if isinstance(data, list):
            return pd.DataFrame(data) if isinstance(data[0], dict) else pd.DataFrame({"name": data})
        if isinstance(data, dict):
            return pd.DataFrame([data])
        return pd.DataFrame({"raw": [data]})
    except Exception as e:
        print(f"fetch_tags unexpected error: {type(e).__name__}: {e}")
        return pd.DataFrame()


def fetch_conesearch_sliced(
    ra: float, dec: float, radius: float, n: int = N_ALERTS_MAX, columns=None
) -> pd.DataFrame:
    """
    Spatially tiled cone-search fallback covering the same sky area as a single
    large-radius conesearch, by summing many small-radius conesearches on a grid.

    Used when a plain fetch_conesearch() call is empty or saturates N_ALERTS_MAX
    (a sign of truncation for dense fields such as COSMOS) -- a single API call
    is capped at N_ALERTS_MAX alerts, so a dense field only returns a partial,
    epoch-biased subset, artificially truncating the observational baseline.
    Identical strategy to 09_Cepheids/02_cepheids_extended_search.ipynb.
    """
    tile_radius = min(TILE_RADIUS_MAX, max(TILE_RADIUS_MIN, radius / 3.5))  # arcsec
    step_arcsec = 0.95 * np.sqrt(2.0) * tile_radius
    nside = int(np.ceil((2.0 * radius) / step_arcsec)) + 1
    if nside % 2 == 0:
        nside += 1
    offs_arcsec = np.linspace(-radius, radius, nside)

    print(
        f"   tiled fallback: nside={nside}, tile_radius={tile_radius:.0f} arcsec, step~{step_arcsec:.0f} arcsec"
    )

    tile_dfs = []
    for dx_as in offs_arcsec:
        for dy_as in offs_arcsec:
            if np.hypot(dx_as, dy_as) > (radius + tile_radius):
                continue
            ra_i = ra + dx_as / 3600.0
            dec_i = dec + dy_as / 3600.0
            dfi = fetch_conesearch(ra_i, dec_i, tile_radius, n, columns)
            if not dfi.empty:
                tile_dfs.append(dfi)
            time.sleep(0.15)

    if not tile_dfs:
        return pd.DataFrame()

    df_all = pd.concat(tile_dfs, ignore_index=True)
    dedup_cols = [
        c for c in ("r:diaSourceId", "r:diaObjectId", "r:midpointMjdTai", "r:band") if c in df_all.columns
    ]
    if dedup_cols:
        df_all = df_all.drop_duplicates(subset=dedup_cols)
    return df_all


print("API wrappers defined.")

## 2b. Fetch Fink Blocks and Tags (informational)

In [ ]:
df_blocks = fetch_blocks()
if not df_blocks.empty:
    print(f"Fink Blocks ({len(df_blocks)} entries):")
    display(df_blocks.T)
else:
    print("No blocks returned (endpoint may not be available in this API version).")

In [ ]:
df_tags = fetch_tags()
if not df_tags.empty:
    print(f"Fink Tags ({len(df_tags)} entries):")
    display(df_tags.T)
else:
    print("No tags returned (endpoint may not be available in this API version).")

## 3. Utility functions

In [ ]:
AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def flux_to_mag(flux_nJy, flux_err_nJy=None):
    """Convert nJy flux (and optional uncertainty) to AB magnitudes."""
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    """Normalised RMS sigma/<f> -- a crude flatness/variability metric."""
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f) / np.mean(f)) if len(f) >= 3 else np.nan


def fractional_variability(flux, flux_err):
    """
    Excess variance normalised fractional variability F_var (Vaughan et al. 2003),
    widely used to characterise AGN variability net of the measurement noise:

        F_var = sqrt( (S^2 - <sigma_err^2>) / <f>^2 )

    where S^2 is the sample variance of the flux and <sigma_err^2> the mean squared
    measurement uncertainty. Returns NaN if the excess variance is negative (i.e.
    the light curve is consistent with pure noise -- not variable within errors) or
    if there are too few points.
    """
    f = np.asarray(flux, dtype=float)
    e = np.asarray(flux_err, dtype=float)
    mask = np.isfinite(f) & np.isfinite(e) & (f > 0)
    f, e = f[mask], e[mask]
    if len(f) < 5:
        return np.nan
    mean_f = np.mean(f)
    s2 = np.var(f, ddof=1)
    mean_err2 = np.mean(e**2)
    excess = s2 - mean_err2
    if excess <= 0 or mean_f <= 0:
        return np.nan
    return float(np.sqrt(excess) / mean_f)


def parse_dipole_bool(series: pd.Series) -> pd.Series:
    """Convert a dipole boolean column (bool, int, or string) to bool."""

    def _to_bool(val):
        if isinstance(val, bool):
            return val
        if isinstance(val, (int, float)):
            return bool(val)
        if isinstance(val, str):
            return val.strip().lower() in ("true", "1", "yes")
        return False

    return series.apply(_to_bool)


def filter_lc(
    df_lc,
    mjd_col="r:midpointMjdTai",
    flux_col="r:psfFlux",
    ferr_col="r:psfFluxErr",
    band_col="r:band",
    snr_min=SNR_MIN,
):
    """
    Filter a raw light-curve DataFrame: drop low-SNR points, convert to magnitudes.
    Works for both diaSources and forced photometry DataFrames.
    Dipole columns (r:isDipole, r:isNegative, ...) are preserved if present.
    """
    df = df_lc.copy()
    for col in (mjd_col, flux_col, ferr_col, band_col):
        if col not in df.columns:
            return pd.DataFrame()
    df[flux_col] = pd.to_numeric(df[flux_col], errors="coerce")
    df[ferr_col] = pd.to_numeric(df[ferr_col], errors="coerce")
    df[mjd_col] = pd.to_numeric(df[mjd_col], errors="coerce")
    snr = df[flux_col].abs() / df[ferr_col].replace(0, np.nan)
    df = df[snr >= snr_min].sort_values(mjd_col).reset_index(drop=True)
    df = df.dropna(subset=[flux_col, ferr_col, mjd_col]).reset_index(drop=True)
    mag, mag_err = flux_to_mag(df[flux_col].values, df[ferr_col].values)
    df["mag"] = mag
    df["mag_err"] = mag_err
    df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)
    return df


def mjd_to_isot(mjd):
    """Convert an MJD float to an ISO date string YYYY-MM-DD."""
    try:
        return Time(mjd, format="mjd").isot[:10]
    except Exception:
        return str(mjd)


print("Utility functions defined.")

## 4. AGN classification function

This is the crux of the notebook. AGN light curves cannot be selected with a single
crossmatch column the way Cepheids can (a dedicated `Cepheid` SIMBAD/VSX type exists),
because:

- **No single catalogue is complete.** Only the brightest, most extreme AGN are
  Fermi-LAT gamma-ray blazars (4LAC) or X-ray-selected blazars (3HSP). The vast
  majority of AGN in a DDF have no dedicated "AGN" crossmatch flag and must be
  recognised through their **host galaxy** (SIMBAD `AGN`/`QSO`/`Sy1`/`Sy2` types,
  or a Legacy-DR8 galaxy classification) or through their **photometric behaviour**
  (Fink's CATS ML classifier, trained to separate periodic/non-periodic/transient
  classes).
- **AGN variability is intrinsically weak and stochastic**, unlike the high-amplitude
  periodic signal of a Cepheid or RR Lyrae. A short or sparsely sampled light curve
  is simply *not usable* for AGN science even if the crossmatch confirms the source
  is an AGN -- so the classification step (§4) and the light-curve-suitability cuts
  (§6, on baseline + number of points) are two separate, complementary filters.
- **Confusion with point-like variable stars and SNe must be avoided.** A stochastic,
  long-duration light curve without periodicity is also the observational signature
  of some evolved-star and SN types, so we explicitly VETO objects with high-
  confidence Gaia-variable, VSX/GCVS periodic, TNS supernova, or Solar-System
  crossmatches, keeping only genuinely AGN-like or unclassified-but-CATS-non-periodic
  candidates.

### Selection tiers (priority order, first match wins)

| Tier | Group | Evidence |
|------|-------|----------|
| 1 | `agn_4lac_gammaray` | 4LAC Fermi-LAT gamma-ray AGN/blazar crossmatch (highest confidence) |
| 2 | `agn_3hsp_blazar` | 3HSP High-Synchrotron-Peaked blazar crossmatch |
| 3 | `agn_simbad_confirmed` | SIMBAD otype in a confirmed AGN/QSO/Seyfert/blazar set |
| 4 | `agn_simbad_candidate` | SIMBAD otype in an AGN-*candidate* set (`AGN_Candidate`, `QSO_Candidate`, ...) |
| 5 | `agn_tns_flare` | TNS classification containing `AGN` (rare, e.g. changing-look AGN / AGN flares) |
| 6 | `agn_cats_nonperiodic_hostgalaxy` | Fink CATS classifier predicts *Non-periodic* (`f:clf_cats_class == 22`) with high score **and** a crossmatched host galaxy (Mangrove/HyperLEDA or Legacy-DR8 low-P(star)) -- the best proxy for *photometrically AGN-like, galaxy-hosted* objects without an explicit AGN catalogue match |
| 7 | `agn_cats_nonperiodic_pointlike` | CATS Non-periodic with high score but no galaxy counterpart -- weaker candidates (could be an unresolved/high-z AGN, or residual contamination); kept separately so they can be excluded easily downstream |
| -- | `not_agn` | everything else (explicitly vetoed or unclassified) |

### Explicit vetoes (applied before all tiers)

An object is **never** classified as AGN if it matches any of the following, since
these indicate a different, well-identified physical origin for the variability:
- `f:is_sso` True (Solar System object)
- non-null `f:xm_tns_fullname` **and** the TNS type does *not* contain `"AGN"`
  (i.e. it is a confirmed SN/other transient)
- SIMBAD otype is a stellar or periodic-variable type (`*`, `RRLyr`, `Cep`, `EB*`, ...)
- Gaia DR3 `VarFlag` indicates variability **and** the Legacy-DR8 P(star) is high
  (> 0.8) -- i.e. a confirmed *stellar* variable, not a galaxy nucleus

In [ ]:
# -- SIMBAD otype sets --------------------------------------------------------
# Confirmed AGN / QSO / Seyfert / blazar SIMBAD object types.
SIMBAD_AGN_CONFIRMED = {
    "AGN",
    "QSO",
    "Sy1",
    "Sy2",
    "Sy",
    "Bla",
    "BLL",
    "BlL",
    "LIN",
    "rG",
    "Sy1_Candidate",
}
# AGN *candidate* SIMBAD object types (lower confidence).
SIMBAD_AGN_CANDIDATE = {
    "AGN_Candidate",
    "QSO_Candidate",
    "Bla_Candidate",
    "EmG",
}
# Stellar / periodic-variable SIMBAD types used only as a VETO (never AGN).
SIMBAD_STELLAR_VETO = {
    "*",
    "**",
    "SB*",
    "PM*",
    "V*",
    "RGB*",
    "LP*",
    "Ev*",
    "Em*",
    "WR*",
    "Be*",
    "BS*",
    "S*b",
    "HB*",
    "sg*",
    "Cep",
    "RRLyr",
    "EB*",
    "Mira",
    "LPV*",
    "SRS",
    "RotV*",
    "EllipV*",
    "RS*",
    "BY*",
    "Fl*",
    "YSO",
    "TTau*",
    "HerbObj",
    "pMS*",
    "Ae*",
    "bCep",
    "SX*",
    "dS*",
    "WVir",
    "RV*",
    "deltaCep",
    "gammaDor",
    "roAp",
}

# -- Fink CATS ML classifier -------------------------------------------------
# f:clf_cats_class: -1=not processed, 11=SN-like, 12=Fast, 13=Long,
#                    21=Periodic, 22=Non-periodic (e.g. AGN).
# See Leoni et al. 2024, https://arxiv.org/abs/2404.08798
CATS_CLASS_NONPERIODIC = 22
CATS_SCORE_MIN = 0.5  # minimum classifier probability to trust the class

# -- Null-value sentinel handling --------------------------------------------
NULL_VALS = {"", "None", "nan", "Fail", "null", "NaN"}


def _is_null(val) -> bool:
    return val is None or str(val).strip() in NULL_VALS


def _to_bool(val) -> bool:
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)):
        return bool(val)
    if isinstance(val, str):
        return val.strip().lower() in ("true", "1", "yes")
    return False


def classify_agn(row: dict) -> tuple[str, str]:
    """
    Classify a diaObject as an AGN candidate (or not) using Fink crossmatch and
    ML-classifier columns aggregated over all its alerts (see aggregate_xm_for_object).

    Returns
    -------
    (group, evidence) : tuple[str, str]
        group    -- one of the tiers described in the markdown cell above
                    ("not_agn" if none of the AGN criteria are met, or if a veto fires)
        evidence -- the catalogue identifier / type string supporting the classification
                    (or None)
    """
    evidence = None

    # -- Read relevant columns -------------------------------------------------
    is_sso = _to_bool(row.get("f:is_sso", False))
    simbad = str(row.get("f:xm_simbad_otype", "") or "").strip()
    x3hsp = str(row.get("f:xm_x3hsp_type", "") or "").strip()
    x4lac = str(row.get("f:xm_x4lac_type", "") or "").strip()
    tns_name = str(row.get("f:xm_tns_fullname", "") or "").strip()
    tns_type = str(row.get("f:xm_tns_type", "") or "").strip()
    mangrove = str(row.get("f:xm_mangrove_2MASS_name", "") or "").strip()
    hypleda = str(row.get("f:xm_mangrove_HyperLEDA_name", "") or "").strip()
    legacy_pstar = row.get("f:xm_legacydr8_pstar", None)
    gaia_var = str(row.get("f:xm_gaiadr3_VarFlag", "") or "").strip()
    gaia_name = str(row.get("f:xm_gaiadr3_DR3Name", "") or "").strip()
    cats_class = row.get("f:clf_cats_class", None)
    cats_score = row.get("f:clf_cats_score", None)

    # ── VETO 0: Solar System objects ──────────────────────────────────────────
    if is_sso:
        return "not_agn", evidence

    # ── VETO 1: confirmed non-AGN TNS transient (SN, nova, ...) ──────────────
    if not _is_null(tns_name) and "AGN" not in tns_type.upper():
        return "not_agn", evidence

    # ── VETO 2: confirmed stellar / periodic-variable SIMBAD type ────────────
    if simbad in SIMBAD_STELLAR_VETO:
        return "not_agn", evidence

    # ── VETO 3: confirmed Gaia stellar variable with high P(star) ────────────
    is_gaia_var = not _is_null(gaia_var) and gaia_var not in ("0", "False", "NOT_AVAILABLE")
    if is_gaia_var and not _is_null(gaia_name):
        try:
            if legacy_pstar is not None and float(legacy_pstar) > 0.8:
                return "not_agn", evidence
        except (TypeError, ValueError):
            pass

    # ── TIER 1: Fermi-LAT gamma-ray AGN (4LAC) -- highest confidence ─────────
    if not _is_null(x4lac):
        return "agn_4lac_gammaray", x4lac

    # ── TIER 2: 3HSP high-synchrotron-peaked blazar ──────────────────────────
    if not _is_null(x3hsp):
        return "agn_3hsp_blazar", x3hsp

    # ── TIER 3: confirmed AGN/QSO/Seyfert/blazar SIMBAD type ─────────────────
    if simbad in SIMBAD_AGN_CONFIRMED:
        return "agn_simbad_confirmed", simbad

    # ── TIER 4: AGN-candidate SIMBAD type ─────────────────────────────────────
    if simbad in SIMBAD_AGN_CANDIDATE:
        return "agn_simbad_candidate", simbad

    # ── TIER 5: TNS classification explicitly mentioning AGN ─────────────────
    if not _is_null(tns_name) and "AGN" in tns_type.upper():
        return "agn_tns_flare", tns_name

    # ── TIER 6/7: Fink CATS non-periodic classifier ───────────────────────────
    try:
        is_nonperiodic = (
            cats_class is not None
            and int(float(cats_class)) == CATS_CLASS_NONPERIODIC
            and cats_score is not None
            and float(cats_score) >= CATS_SCORE_MIN
        )
    except (TypeError, ValueError):
        is_nonperiodic = False

    if is_nonperiodic:
        has_host_galaxy = not _is_null(hypleda) or not _is_null(mangrove)
        try:
            has_host_galaxy = has_host_galaxy or (legacy_pstar is not None and float(legacy_pstar) < 0.2)
        except (TypeError, ValueError):
            pass

        if has_host_galaxy:
            return "agn_cats_nonperiodic_hostgalaxy", hypleda or mangrove or "legacy_galaxy"
        else:
            return "agn_cats_nonperiodic_pointlike", None

    # ── No AGN evidence ────────────────────────────────────────────────────────
    return "not_agn", evidence


# Groups that are considered genuine AGN light-curve candidates for downloading.
AGN_GROUPS = [
    "agn_4lac_gammaray",
    "agn_3hsp_blazar",
    "agn_simbad_confirmed",
    "agn_simbad_candidate",
    "agn_tns_flare",
    "agn_cats_nonperiodic_hostgalaxy",
    "agn_cats_nonperiodic_pointlike",
]

print("classify_agn() defined.")
print(f"AGN_GROUPS ({len(AGN_GROUPS)}): {AGN_GROUPS}")

## 5. Cone searches on the Deep Drilling Fields

**Aggregation strategy** (as in `04_calib` and `09_Cepheids`): `f:xm_*` and
`f:clf_*` columns can be null or inconsistent on individual alerts of the same
object. We aggregate all alerts per `diaObjectId` and take the **mode of the
non-null values** for string/crossmatch columns before running `classify_agn()`,
and the **mean** for the CATS score.

**Extended (tiled) search for dense fields**: a single `conesearch` call is
capped at `N_ALERTS_MAX` alerts. Dense fields -- **COSMOS** in particular --
saturate this cap, so a plain call only returns a partial, epoch-biased subset
of alerts, which artificially truncates `nDiaSources` and, critically, the
**observational baseline** (`firstDiaSourceMjdTai`/`lastDiaSourceMjdTai`) used
by the AGN light-curve-suitability cuts in section 6. For each field we first
try a plain `conesearch`; if it is empty **or** returns `>= SATURATION_FRAC *
N_ALERTS_MAX` alerts (a sign of truncation), we fall back to
`fetch_conesearch_sliced()`, a spatially tiled search that sums many
small-radius conesearches covering the same sky area -- identical strategy to
`09_Cepheids/02_cepheids_extended_search.ipynb`. Results are cached per field
as Parquet in `DIR_DATA` so the (slower) tiled search only runs once.

In [ ]:
# -- Columns to fetch from conesearch -----------------------------------------
COLS_CONE = (
    "r:diaObjectId,r:diaSourceId,r:nDiaSources,r:ra,r:dec,r:band,r:midpointMjdTai,"
    "r:psfFlux,r:psfFluxErr,r:extendedness,"
    "r:target_name,"
    "r:isNegative,r:isDipole,r:dipoleFitAttempted,"
    "r:dipoleFluxDiff,r:dipoleFluxDiffErr,r:dipoleMeanFlux,r:dipoleMeanFluxErr,"
    "r:dipoleLength,r:dipoleAngle,r:dipoleNdata,r:dipoleChi2,"
    "r:visit,r:detector,r:x,r:y,r:xErr,r:yErr,"
    # Gaia DR3
    "f:xm_gaiadr3_DR3Name,f:xm_gaiadr3_VarFlag,f:xm_gaiadr3_Plx,f:xm_gaiadr3_e_Plx,"
    # SIMBAD
    "f:xm_simbad_otype,"
    # Legacy DR8 (star/galaxy separator + photo-z, key for AGN host-galaxy evidence)
    "f:xm_legacydr8_pstar,f:xm_legacydr8_zphot,f:xm_legacydr8_fqual,"
    # Mangrove (nearby galaxies -- AGN host-galaxy evidence)
    "f:xm_mangrove_2MASS_name,f:xm_mangrove_HyperLEDA_name,"
    # Variable star catalogues (used only as vetoes)
    "f:xm_vsx_Type,f:xm_gcvs_type,"
    # Young Stellar Objects (veto)
    "f:xm_spicy_class,"
    # Transient Name Server
    "f:xm_tns_fullname,f:xm_tns_type,"
    # High-energy blazars (3HSP + 4LAC) -- primary AGN evidence
    "f:xm_x3hsp_type,f:xm_x4lac_type,"
    # Fink flags
    "f:is_sso,f:is_cataloged,"
    # Fink ML classifiers -- CATS non-periodic class is central to AGN selection
    "f:clf_cats_class,f:clf_cats_score,f:clf_snnSnVsOthers_score,"
    "f:main_label_classifier,f:main_label_crossmatch,"
)

# Crossmatch columns subject to alert-level inconsistency (aggregated by mode).
XM_COLS = [
    "f:xm_gaiadr3_DR3Name",
    "f:xm_gaiadr3_VarFlag",
    "f:xm_gaiadr3_Plx",
    "f:xm_gaiadr3_e_Plx",
    "f:xm_simbad_otype",
    "f:xm_legacydr8_pstar",
    "f:xm_legacydr8_zphot",
    "f:xm_legacydr8_fqual",
    "f:xm_mangrove_2MASS_name",
    "f:xm_mangrove_HyperLEDA_name",
    "f:xm_vsx_Type",
    "f:xm_gcvs_type",
    "f:xm_spicy_class",
    "f:xm_tns_fullname",
    "f:xm_tns_type",
    "f:xm_x3hsp_type",
    "f:xm_x4lac_type",
    "f:clf_cats_class",
    "f:main_label_classifier",
    "f:main_label_crossmatch",
]
# Boolean Fink flags -- majority vote, NOT string mode (bool('False') == True!).
BOOL_COLS = ["f:is_sso", "f:is_cataloged"]
# Numeric columns aggregated by mean (classifier scores).
NUMERIC_MEAN_COLS = ["f:clf_cats_score", "f:clf_snnSnVsOthers_score"]


def aggregate_xm_for_object(df_alerts: pd.DataFrame) -> dict:
    """
    Given all conesearch alerts for one object, return a single representative
    dict of crossmatch/classifier values:
      - XM_COLS            -> mode of non-null values across alerts
      - BOOL_COLS           -> majority vote
      - NUMERIC_MEAN_COLS   -> mean of non-null values
    Also computes per-object dipole statistics (fraction, count, any-flag).
    """
    best = {}
    for col in XM_COLS:
        if col not in df_alerts.columns:
            best[col] = None
            continue
        vals = df_alerts[col].dropna().astype(str)
        vals = vals[~vals.isin(NULL_VALS)]
        best[col] = vals.mode().iloc[0] if not vals.empty else None

    for col in BOOL_COLS:
        if col not in df_alerts.columns:
            best[col] = False
            continue
        bvals = df_alerts[col].dropna()
        best[col] = (
            False
            if bvals.empty
            else bool(bvals.apply(lambda v: str(v).strip().lower() in ("true", "1", "yes")).mean() > 0.5)
        )

    for col in NUMERIC_MEAN_COLS:
        if col not in df_alerts.columns:
            best[col] = np.nan
            continue
        vals = pd.to_numeric(df_alerts[col], errors="coerce").dropna()
        best[col] = float(vals.mean()) if not vals.empty else np.nan

    # -- Per-object dipole statistics (aggregated over all alerts) ------------
    if "r:isDipole" in df_alerts.columns:
        is_dipole_series = parse_dipole_bool(df_alerts["r:isDipole"].fillna(False))
        n_total = len(df_alerts)
        n_dipole = int(is_dipole_series.sum())
        best["dipole_ndetect"] = n_dipole
        best["dipole_fraction"] = n_dipole / n_total if n_total > 0 else 0.0
        best["isDipole_any"] = n_dipole > 0
    else:
        best["dipole_ndetect"] = 0
        best["dipole_fraction"] = 0.0
        best["isDipole_any"] = False

    return best


all_candidates = {}  # diaObjectId (str) -> metadata dict

# loop on DDF to fetch the alerts
for field_name, (ra, dec) in DEEP_FIELDS.items():
    print(f"\n-- Cone search: {field_name}  RA={ra:.4f}  Dec={dec:.4f}  r={CONE_RADIUS:.0f} arcsec")

    cache_path = os.path.join(DIR_DATA, f"conesearch_{field_name.replace('-', '_')}.parquet")
    if os.path.exists(cache_path) and not FORCE_RELOAD_CONE:
        df_cone = pd.read_parquet(cache_path)
        print(f"  [CACHE] loaded {len(df_cone)} alerts from {cache_path}")
    else:
        try:
            df_cone = fetch_conesearch(ra, dec, CONE_RADIUS, n=N_ALERTS_MAX, columns=COLS_CONE)
        except Exception as e:
            print(f"  ERROR: {e}")
            continue

        is_saturated = len(df_cone) >= SATURATION_FRAC * N_ALERTS_MAX
        if df_cone.empty or is_saturated:
            reason = (
                "no data" if df_cone.empty else f"saturated ({len(df_cone)} >= {SATURATION_FRAC:.0%} of cap)"
            )
            print(f"  Plain conesearch {reason} -- falling back to tiled extended search...")
            df_tiled = fetch_conesearch_sliced(ra, dec, CONE_RADIUS, n=N_ALERTS_MAX, columns=COLS_CONE)
            if not df_tiled.empty:
                df_cone = df_tiled
                print(f"  Tiled search recovered {len(df_cone)} alerts.")

        if not df_cone.empty:
            df_cone.to_parquet(cache_path, index=False)
            print(f"  Saved cache: {cache_path}")

    if df_cone.empty:
        print("  No alerts returned.")
        continue

    print(f"  Alerts returned: {len(df_cone)}")

    oid_col = "r:diaObjectId"
    nsrc_col = "r:nDiaSources"

    if oid_col not in df_cone.columns:
        print(f"  Column {oid_col!r} missing -- cols: {df_cone.columns.tolist()[:10]}")
        continue

    if nsrc_col in df_cone.columns:
        df_cone[nsrc_col] = pd.to_numeric(df_cone[nsrc_col], errors="coerce").fillna(0)

    grouped = df_cone.groupby(oid_col)
    df_nsrc = df_cone.groupby(oid_col)[nsrc_col].max().reset_index() if nsrc_col in df_cone.columns else None
    df_pos = df_cone.groupby(oid_col)[["r:ra", "r:dec"]].first().reset_index()

    print(f"  Now aggregate the rows: {len(grouped)}")
    aggregated_rows = []
    for oid, grp in grouped:
        row = {oid_col: oid}
        row.update(aggregate_xm_for_object(grp))
        aggregated_rows.append(row)

    print(f"  Now create df_obj")
    print(f"aggregated_rows size: {len(aggregated_rows)}")
    print(f"example keys: {list(aggregated_rows[0].keys())[:10]}")
    print(f"approx memory: {sys.getsizeof(aggregated_rows)}")

    df_obj = pd.DataFrame(aggregated_rows)
    if df_nsrc is not None:
        df_obj = df_obj.merge(df_nsrc, on=oid_col, how="left")
    df_obj = df_obj.merge(df_pos, on=oid_col, how="left")

    print(f"  Unique objects: {len(df_obj)}")

    df_ok = df_obj[df_obj[nsrc_col] >= NP_MIN] if nsrc_col in df_obj.columns else df_obj
    print(f"  Objects with >={NP_MIN} detections: {len(df_ok)}")

    for _, row in df_ok.iterrows():
        oid = str(row[oid_col])
        nsrc = int(row.get(nsrc_col, 0))
        group, evidence = classify_agn(row.to_dict())
        if oid not in all_candidates:
            all_candidates[oid] = {
                "nDiaSources": nsrc,
                "group": group,
                "evidence": evidence,
                "field": field_name,
                "ra": float(row.get("r:ra", float("nan"))),
                "dec": float(row.get("r:dec", float("nan"))),
                "xm_simbad_otype": row.get("f:xm_simbad_otype"),
                "xm_x3hsp_type": row.get("f:xm_x3hsp_type"),
                "xm_x4lac_type": row.get("f:xm_x4lac_type"),
                "xm_mangrove_2MASS_name": row.get("f:xm_mangrove_2MASS_name"),
                "xm_mangrove_HyperLEDA_name": row.get("f:xm_mangrove_HyperLEDA_name"),
                "xm_legacydr8_pstar": row.get("f:xm_legacydr8_pstar"),
                "xm_legacydr8_zphot": row.get("f:xm_legacydr8_zphot"),
                "xm_tns_fullname": row.get("f:xm_tns_fullname"),
                "xm_tns_type": row.get("f:xm_tns_type"),
                "clf_cats_class": row.get("f:clf_cats_class"),
                "clf_cats_score": row.get("f:clf_cats_score"),
                "is_sso": row.get("f:is_sso"),
                "dipole_ndetect": row.get("dipole_ndetect", 0),
                "dipole_fraction": row.get("dipole_fraction", 0.0),
                "isDipole_any": row.get("isDipole_any", False),
            }

    time.sleep(0.5)

print(f"\n=== Total unique candidates (all fields, nDiaSources >= {NP_MIN}): {len(all_candidates)} ===")

group_counts = collections.Counter(v["group"] for v in all_candidates.values())
print("\nGroup distribution:")
for g, n in sorted(group_counts.items(), key=lambda x: -x[1]):
    print(f"  {g:35s} {n:5d} objects")

## 6. Build the AGN candidate catalogue and apply light-curve-suitability cuts

### Why "long duration, many points" matters for AGN

A crossmatch confirming an object *is* an AGN says nothing about whether its
**Fink/LSST light curve is usable** for AGN variability science. Two additional,
purely *photometric* cuts are applied on top of the `classify_agn()` tiers:

1. **`nDiaSources >= NP_MIN_AGN`** -- enough independent epochs to measure a
   meaningful structure function / excess variance instead of being dominated by
   Poisson noise from a handful of points.
2. **`baseline_days = lastDiaSourceMjdTai - firstDiaSourceMjdTai >= MIN_BASELINE_DAYS`**
   -- AGN optical variability is a **stochastic (red-noise) process** with a
   characteristic correlation/damping timescale typically from tens to a few
   hundred days (e.g. the damped-random-walk timescale of MacLeod et al. 2010).
   A light curve *shorter* than this timescale cannot distinguish AGN-like
   red-noise variability from white noise or from a single flare/artefact -- the
   variance keeps growing with the sampled baseline until it saturates near the
   damping timescale, so **duration, not just point count, is the limiting factor**.

Both quantities come from `/api/v1/objects` (`r:nDiaSources`,
`r:firstDiaSourceMjdTai`, `r:lastDiaSourceMjdTai`), fetched in batches of 50
`diaObjectId` per call for efficiency.

In [ ]:
catalogue_rows = []
for oid, meta in all_candidates.items():
    row = {"diaObjectId": oid, **meta}
    catalogue_rows.append(row)

df_agn_catalogue = pd.DataFrame(catalogue_rows)
print(f"Base AGN-candidate catalogue: {len(df_agn_catalogue)} objects")
print()
print("Objects per group:")
print(df_agn_catalogue["group"].value_counts().to_string())
print()
print("Objects per DDF:")
print(df_agn_catalogue["field"].value_counts().to_string())
display(df_agn_catalogue.head(5))

In [ ]:
# -- Fetch /api/v1/objects data in batches for all candidates -----------------
BATCH_SIZE = 50
SLEEP_BETWEEN_BATCHES = 0.5

all_oids = df_agn_catalogue["diaObjectId"].astype(str).tolist()
objects_rows = []
n_positional_ok = 0
n_positional_mismatch = 0

print(f"Fetching /api/v1/objects for {len(all_oids)} diaObjects in batches of {BATCH_SIZE}...")

for batch_start in range(0, len(all_oids), BATCH_SIZE):
    batch = all_oids[batch_start : batch_start + BATCH_SIZE]
    df_batch = fetch_objects_batch(batch)
    if not df_batch.empty:
        # -- Robust ID matching --------------------------------------------------
        # Large diaObjectId values (18 digits) can lose precision if the API
        # serialises them as JSON floats, which silently breaks a later merge
        # on the returned ID column (this is what caused baseline_days to be
        # NaN for every object in an earlier version of this notebook). Since
        # we know the exact ordered list of IDs we requested, use POSITIONAL
        # matching whenever the API preserves request order (row count equals
        # the batch size) -- this sidesteps ID precision loss entirely.
        if len(df_batch) == len(batch):
            df_batch = df_batch.copy()
            df_batch["diaObjectId_requested"] = batch
            n_positional_ok += len(batch)
        else:
            n_positional_mismatch += len(batch)
        objects_rows.append(df_batch)
    n_done = min(batch_start + BATCH_SIZE, len(all_oids))
    if (batch_start // BATCH_SIZE) % 5 == 0:
        print(f"  {n_done}/{len(all_oids)} fetched...")
    time.sleep(SLEEP_BETWEEN_BATCHES)

if objects_rows:
    df_objects_api = pd.concat(objects_rows, ignore_index=True)
    print(f"\nAPI objects table shape : {df_objects_api.shape}")
    print(f"Objects matched positionally (robust) : {n_positional_ok}")
    if n_positional_mismatch:
        print(
            f"WARNING: {n_positional_mismatch} objects came from batches where the API returned a "
            f"different row count than requested -- these fall back to ID-column matching (less robust)."
        )
else:
    df_objects_api = pd.DataFrame()
    print("WARNING: /api/v1/objects returned no data -- catalogue will not be enriched.")

In [ ]:
pd.set_option("display.max_rows", 100)
summary_bad_obj_cols = pd.DataFrame(
    {
        "non_null": df_objects_api.notna().sum(),
        "null": df_objects_api.isna().sum(),
        "fraction_null": df_objects_api.isna().mean(),
        "dtype": df_objects_api.dtypes,
    }
).sort_values("fraction_null", ascending=False)

In [ ]:
display(summary_bad_obj_cols)

In [ ]:
df_objects_api.head()

In [ ]:
if not df_objects_api.empty:
    obj_id_col = None
    for candidate in ("i:diaObjectId", "diaObjectId", "f:diaObjectId", "r:diaObjectId"):
        if candidate in df_objects_api.columns:
            obj_id_col = candidate
            print(f"objectOd columns = {obj_id_col}")
            break

    df_objects_api_clean = df_objects_api.copy()

    if "diaObjectId_requested" in df_objects_api_clean.columns:
        # Prefer the robust, positionally-assigned ID (see fetch cell above) --
        # avoids precision-loss issues with large diaObjectId values that can
        # silently break a merge on the API's own returned ID column.
        df_objects_api_clean["diaObjectId"] = df_objects_api_clean["diaObjectId_requested"]
        print("Using positional diaObjectId matching (robust to ID precision loss).")
    elif obj_id_col is not None:
        df_objects_api_clean = df_objects_api_clean.rename(columns={obj_id_col: "diaObjectId"})
        df_objects_api_clean["diaObjectId"] = df_objects_api_clean["diaObjectId"].astype(str)
        print(f"Using API-returned ID column {obj_id_col!r} for matching (fallback -- less robust).")
    else:
        print("ERROR: cannot find any diaObjectId column in /objects response.")
        df_objects_api_clean = pd.DataFrame()

    if not df_objects_api_clean.empty:
        df_agn_enriched = df_agn_catalogue.merge(
            df_objects_api_clean, on="diaObjectId", how="left", suffixes=("", "_obj")
        )
        print(f"Enriched catalogue shape : {df_agn_enriched.shape}")
    else:
        df_agn_enriched = df_agn_catalogue.copy()
else:
    df_agn_enriched = df_agn_catalogue.copy()
    print("No /api/v1/objects data -- using base catalogue only.")

# -- Locate first/last detection MJD columns (naming may vary with API version) --
first_mjd_col = next((c for c in df_agn_enriched.columns if c.endswith("firstDiaSourceMjdTai")), None)
last_mjd_col = next((c for c in df_agn_enriched.columns if c.endswith("lastDiaSourceMjdTai")), None)
nsrc_api_col = next(
    (c for c in df_agn_enriched.columns if c.endswith("nDiaSources") and c != "nDiaSources"), None
)

print(f"first-MJD column : {first_mjd_col}")
print(f"last-MJD column  : {last_mjd_col}")
print(f"nDiaSources (API): {nsrc_api_col}")

if first_mjd_col and last_mjd_col:
    df_agn_enriched["firstDiaSourceMjdTai"] = pd.to_numeric(df_agn_enriched[first_mjd_col], errors="coerce")
    df_agn_enriched["lastDiaSourceMjdTai"] = pd.to_numeric(df_agn_enriched[last_mjd_col], errors="coerce")
    df_agn_enriched["baseline_days"] = (
        df_agn_enriched["lastDiaSourceMjdTai"] - df_agn_enriched["firstDiaSourceMjdTai"]
    )
else:
    print("WARNING: first/last MJD columns not found -- baseline_days set to NaN.")
    df_agn_enriched["baseline_days"] = np.nan

if nsrc_api_col:
    df_agn_enriched["nDiaSources_api"] = pd.to_numeric(df_agn_enriched[nsrc_api_col], errors="coerce")
else:
    df_agn_enriched["nDiaSources_api"] = df_agn_enriched["nDiaSources"]

print(f"\nBaseline (days) stats:\n{df_agn_enriched['baseline_days'].describe()}")

In [ ]:
# -- Apply the AGN light-curve-suitability cuts --------------------------------
is_agn_class = df_agn_enriched["group"].isin(AGN_GROUPS)
is_well_sampled = df_agn_enriched["nDiaSources_api"].fillna(df_agn_enriched["nDiaSources"]) >= NP_MIN_AGN
is_long_baseline = df_agn_enriched["baseline_days"] >= MIN_BASELINE_DAYS

# mjd-first and mjd-last not defined : this baseline_days does not exists
# df_agn_selected = df_agn_enriched[is_agn_class & is_well_sampled & is_long_baseline].copy()
# df_agn_selected = df_agn_selected.sort_values("baseline_days", ascending=False).reset_index(drop=True)

# keep only two criteria
df_agn_selected = df_agn_enriched[is_agn_class & is_well_sampled].copy()
df_agn_selected = df_agn_selected.sort_values("nDiaSources", ascending=False).reset_index(drop=True)

print(f"AGN-classified objects           : {int(is_agn_class.sum())}")
print(f"  ... with nDiaSources >= {NP_MIN_AGN:<4d}  : {int((is_agn_class & is_well_sampled).sum())}")
print(f"  ... AND baseline >= {MIN_BASELINE_DAYS:.0f} d : {len(df_agn_selected)}")
print()
print("Selected AGN candidates per group:")
print(df_agn_selected["group"].value_counts().to_string())
print()
print("Selected AGN candidates per field:")
print(df_agn_selected["field"].value_counts().to_string())

display(
    df_agn_selected[
        [
            "diaObjectId",
            "group",
            "evidence",
            "field",
            "ra",
            "dec",
            "nDiaSources_api",
            "baseline_days",
            "xm_simbad_otype",
            "clf_cats_class",
            "clf_cats_score",
        ]
    ].head(20)
)

# -- Save catalogues ------------------------------------------------------------
cat_path = os.path.join(DIR_DATA, "agn_catalogue_all_candidates.csv")
df_agn_enriched.to_csv(cat_path, index=False)
print(f"\nSaved: {cat_path}  ({len(df_agn_enriched)} rows)")

sel_path = os.path.join(DIR_DATA, "agn_catalogue_selected.csv")
df_agn_selected.to_csv(sel_path, index=False)
print(f"Saved: {sel_path}  ({len(df_agn_selected)} rows)")

## 7. Download light curves for all selected AGN candidates

For every object in `df_agn_selected` we download:
- **diaSources** (`src`) via `/api/v1/sources` -- direct detections, including the
  dipole columns used to flag (not remove) subtraction artefacts
- **Forced photometry** (`fp`) via `/api/v1/fp` -- densifies the light curve at
  every visit, whether or not a significant alert was issued (crucial for AGN,
  whose faint variability often falls below the per-visit detection threshold)

In [ ]:
COLS_SRC = (
    "r:diaObjectId,r:diaSourceId,r:midpointMjdTai,"
    "r:psfFlux,r:psfFluxErr,r:band,r:ra,r:dec,r:snr,r:psfChi2,"
    "r:apFlux,r:apFluxErr,"
    "r:visit,r:detector,r:x,r:y,r:xErr,r:yErr,"
    "r:isDipole,r:isNegative,r:dipoleFitAttempted,"
    "r:dipoleFluxDiff,r:dipoleFluxDiffErr,r:dipoleMeanFlux,r:dipoleMeanFluxErr,"
    "r:dipoleLength,r:dipoleAngle,r:dipoleNdata,r:dipoleChi2,"
    "r:scienceFlux,r:scienceFluxErr,r:templateFlux,r:templateFluxErr"
)

COLS_FP = (
    "r:diaObjectId,r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:band,"
    "r:visit,r:detector,r:x,r:y,r:xErr,r:yErr,"
    "r:scienceFlux,r:scienceFluxErr"
)


def _cast_rubin_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Cast Rubin DRP integer columns to nullable Int64 (robust to float API returns)."""
    for col in ("r:visit", "r:detector"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in ("r:x", "r:y", "r:xErr", "r:yErr"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def _cast_dipole_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Cast dipole boolean and numeric columns to appropriate types."""
    for col in ("r:isDipole", "r:isNegative", "r:dipoleFitAttempted"):
        if col in df.columns:
            df[col] = parse_dipole_bool(df[col].fillna(False))
    for col in (
        "r:dipoleFluxDiff",
        "r:dipoleFluxDiffErr",
        "r:dipoleMeanFlux",
        "r:dipoleMeanFluxErr",
        "r:dipoleLength",
        "r:dipoleAngle",
        "r:dipoleNdata",
        "r:dipoleChi2",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


lc_cache = {}  # oid -> {'src': df, 'fp': df, 'group': str, 'field': str, ...}

oids_to_fetch = df_agn_selected["diaObjectId"].tolist()
print(f"Objects to fetch light curves for: {len(oids_to_fetch)}")

for i, oid in enumerate(oids_to_fetch):
    meta = df_agn_selected[df_agn_selected["diaObjectId"] == oid].iloc[0]
    group = meta["group"]
    try:
        df_src = fetch_sources(oid, columns=COLS_SRC)
        df_fp = fetch_fp(oid, columns=COLS_FP)

        df_src = _cast_rubin_cols(df_src)
        df_src = _cast_dipole_cols(df_src)
        df_fp = _cast_rubin_cols(df_fp)

        df_src_f = filter_lc(df_src)
        df_fp_f = filter_lc(df_fp)

        if len(df_src_f) + len(df_fp_f) > 0:
            lc_cache[oid] = {
                "src": df_src_f,
                "fp": df_fp_f,
                "group": group,
                "evidence": meta.get("evidence"),
                "field": meta.get("field"),
                "baseline_days": meta.get("baseline_days"),
                "isDipole_any": meta.get("isDipole_any", False),
            }
            n_dip_src = int(df_src_f["r:isDipole"].sum()) if "r:isDipole" in df_src_f.columns else 0
            print(
                f"  [{i + 1:3d}/{len(oids_to_fetch)}] {oid}  "
                f"src={len(df_src_f):4d} (dipoles={n_dip_src})  fp={len(df_fp_f):4d}  "
                f"group={group} field={meta.get('field', '?')}"
            )
    except Exception as e:
        print(f"  [{i + 1:3d}/{len(oids_to_fetch)}] {oid}  ERROR: {e}")
    time.sleep(0.2)

print(f"\nDownloaded light curves: {len(lc_cache)} objects")

## 8. Build per-group object lists

In [ ]:
all_groups = sorted(set(d["group"] for d in lc_cache.values()))
group_oids = {g: [] for g in all_groups}
for oid, d in lc_cache.items():
    group_oids[d["group"]].append(oid)

print("Group -> object count (downloaded light curves):")
for g in sorted(group_oids, key=lambda x: -len(group_oids[x])):
    print(f"  {g:35s} {len(group_oids[g]):3d}")

## 9. Variability metrics per object and band

For each object/band we combine `src` + `fp` (deduplicated on MJD+band) and compute:
- `rms_var` = sigma/<f>, a simple normalised scatter (same metric as in `04_calib`,
  here expected to be **large** for genuine AGN, unlike for calibration stars)
- `fvar` = fractional variability F_var (Vaughan et al. 2003), which subtracts the
  photometric noise contribution -- the standard AGN-variability figure of merit

In [ ]:
variability_rows = []

for oid, data in lc_cache.items():
    group = data["group"]
    frames = [df for df in (data["fp"], data["src"]) if not df.empty]
    if not frames:
        continue
    df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
    df_all = df_all.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)
    for band in BANDS:
        df_b = df_all[df_all["r:band"] == band]
        if len(df_b) < 5:
            continue
        rms = rms_variability(df_b["r:psfFlux"].values)
        fvar = fractional_variability(df_b["r:psfFlux"].values, df_b["r:psfFluxErr"].values)
        variability_rows.append(
            {
                "group": group,
                "diaObjectId": oid,
                "band": band,
                "n_pts": len(df_b),
                "rms_var": rms,
                "fvar": fvar,
                "mean_flux_nJy": float(df_b["r:psfFlux"].mean()),
                "baseline_days": data.get("baseline_days"),
            }
        )

df_var = pd.DataFrame(variability_rows)

if not df_var.empty:
    print(f"Variability table: {len(df_var)} rows")
    print(df_var.groupby("group")[["n_pts", "rms_var", "fvar"]].median().round(4))
else:
    print("No variability data -- check light curve downloads.")

df_var.to_csv(os.path.join(DIR_DATA, "agn_variability_metrics.csv"), index=False)
print("\nSaved agn_variability_metrics.csv")

## 10. Plot: variability distribution per group

In [ ]:
if df_var.empty:
    print("No data.")
else:
    groups_present = [g for g in all_groups if g in df_var["group"].unique()]
    bands_present = [b for b in BANDS if b in df_var["band"].unique()]
    short_labels = [g.replace("_", "\n") for g in groups_present]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_var[df_var["band"] == band]
        data_per_group = [df_b[df_b["group"] == g]["fvar"].dropna().values for g in groups_present]
        bp = ax.boxplot(data_per_group, labels=short_labels, patch_artist=True, notch=False, showfliers=True)
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=60, labelsize=7)
        ax.set_yscale("log")

    axes[0].set_ylabel("Fractional variability  F_var")
    fig.suptitle("AGN candidates: fractional variability by class", fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig("01_fvar_boxplot_by_group")
    plt.show()

## 11. Plot light curves -- top objects per group (longest baseline first)

**Dipole overlay**: detections flagged `r:isDipole == True` in the diaSources are
overlaid as large light-grey open circles on top of the standard band-colour
markers, so subtraction-artefact contamination remains visible without removing
the points from the light curve.

In [ ]:
def rank_oids(oid_list, nc=NC_PLOT):
    """Rank objects by observational baseline (longest first) -- AGN priority metric."""
    scored = [(o, lc_cache[o].get("baseline_days") or 0) for o in oid_list if o in lc_cache]
    return [o for o, _ in sorted(scored, key=lambda x: -(x[1] if pd.notna(x[1]) else 0))[:nc]]


def getTminTmax(df, df_src, df_fp):
    t = df["r:midpointMjdTai"].values
    t_src = df_src["r:midpointMjdTai"].values
    tmin = t_src.min() if len(t_src) > 0 else t.min()
    tmax = t.max()
    return tmin, tmax


def getYminYmax(df, df_src, df_fp):
    y = df["r:psfFlux"].values
    y_src = df_src["r:psfFlux"].values
    if len(y_src) > 0:
        ymin, ymax = y_src.min(), y_src.max()
    else:
        ymin, ymax = y.min(), y.max()
    magmin, _ = flux_to_mag(ymax)
    magmax, _ = flux_to_mag(ymin)
    return ymin, ymax, magmin, magmax


def plot_lc_grid(oid_list, group, mode="flux", nc=NC_PLOT):
    """Plot a grid of light curves (one row per object, one column per band)."""
    top = rank_oids(oid_list, nc)
    n_obj = len(top)
    if n_obj == 0:
        print(f"  No objects for group {group}.")
        return

    fig, axes = plt.subplots(
        n_obj, len(BANDS), figsize=(2.8 * len(BANDS), 2.6 * n_obj), sharex=False, sharey=False, squeeze=False
    )

    for row_idx, oid in enumerate(top):
        d = lc_cache[oid]
        frames = [df for df in (d.get("fp", pd.DataFrame()), d.get("src", pd.DataFrame())) if not df.empty]
        if not frames:
            continue
        df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        df_all = df_all.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(
            drop=True
        )

        df_fp = d["fp"].drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        df_fp = df_fp.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)
        df_src = d["src"].drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        df_src = df_src.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(
            drop=True
        )

        if df_all.empty:
            continue

        tmin, tmax = getTminTmax(df_all, df_src, df_fp)
        ymin, ymax, magmin, magmax = getYminYmax(df_all, df_src, df_fp)
        deltamag = np.max([magmax - magmin, 3.0])
        centermag = (magmax + magmin) / 2.0
        ymagmin = centermag - deltamag / 2.0
        ymagmax = centermag + deltamag / 2.0

        first_band = -1
        ax_first_band = None
        for col_idx, band in enumerate(BANDS):
            ax = axes[row_idx][col_idx]
            df_b = df_all[df_all["r:band"] == band].sort_values("r:midpointMjdTai")
            if len(df_b) < 3:
                ax.set_visible(False)
                continue

            if first_band == -1:
                first_band = col_idx
                ax_first_band = ax

            if mode == "flux":
                mask = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
            else:
                mask = np.isfinite(df_b["mag"].values) & np.isfinite(df_b["mag_err"].values)
            df_b = df_b[mask].reset_index(drop=True)
            if len(df_b) < 3:
                ax.set_visible(False)
                continue

            t = df_b["r:midpointMjdTai"].values
            dt = t - tmin
            dtmax = dt.max()

            color = BAND_COLORS.get(band, "gray")
            if mode == "flux":
                y, yerr = df_b["r:psfFlux"].values, df_b["r:psfFluxErr"].values
            else:
                y, yerr = df_b["mag"].values, df_b["mag_err"].values
                ax.invert_yaxis()
            ax.errorbar(dt, y, yerr=yerr, fmt="o", ms=3, lw=0.8, elinewidth=0.8, color=color, alpha=0.8)

            if mode == "flux" and "r:isDipole" in df_b.columns:
                dip_mask = parse_dipole_bool(df_b["r:isDipole"].fillna(False))
                if dip_mask.any():
                    ax.scatter(
                        dt[dip_mask.values],
                        y[dip_mask.values],
                        marker="o",
                        s=120,
                        facecolors="none",
                        edgecolors="lightgrey",
                        linewidths=1.5,
                        zorder=5,
                        alpha=0.9,
                        label="dipole" if col_idx == first_band else None,
                    )

            rms = rms_variability(df_b["r:psfFlux"].values)
            ax.set_title(f"{band} N={len(df_b)} rms={rms:.3f}", fontsize=7, pad=2, color=color)
            ax.set_xlabel("Delta t (days)", fontsize=7)
            ax.tick_params(labelsize=7)
            ax.set_xlim(-1.0, dtmax + 1.0)
            if mode == "flux":
                ax.set_ylim(0.0, 1.2 * ymax)
            else:
                ax.set_ylim(ymagmax, ymagmin)

        if ax_first_band is not None:
            field = lc_cache[oid].get("field")
            base = lc_cache[oid].get("baseline_days")
            label = f"{oid}  [{field}]  T={base:.0f}d" if pd.notna(base) else f"{oid}  [{field}]"
            ax_first_band.set_ylabel(f"{label}\n{'flux (nJy)' if mode == 'flux' else 'AB mag'}", fontsize=9)
        else:
            axes[row_idx][0].set_ylabel(f"{oid}\n{'flux (nJy)' if mode == 'flux' else 'AB mag'}", fontsize=9)

    fig.suptitle(f"AGN group: {group}  |  mode={mode}", fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    safe = group.replace("/", "_").replace(" ", "_")
    savefig(f"02_lc_{safe}_{mode}")
    plt.show()


print("Plot functions defined.")

In [ ]:
groups_to_plot = [g for g in AGN_GROUPS if len(group_oids.get(g, [])) >= 1]

for group in groups_to_plot:
    print(f"\n=== {group} ({len(group_oids[group])} objects) ===")
    plot_lc_grid(group_oids[group], group, mode="flux")

In [ ]:
for group in groups_to_plot:
    print(f"\n=== {group} (magnitude) ===")
    plot_lc_grid(group_oids[group], group, mode="mag")

## 12. Sky distribution of selected AGN candidates

In [ ]:
if df_agn_selected.empty:
    print("No AGN candidates to plot.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))

    for fname, (ra, dec) in DEEP_FIELDS.items():
        ax.scatter(ra, dec, marker="+", s=200, color="grey", lw=1.5, zorder=2)
        ax.text(ra + 0.5, dec + 0.3, fname, fontsize=7, color="grey")

    group_markers = {
        "agn_4lac_gammaray": ("*", "crimson"),
        "agn_3hsp_blazar": ("^", "darkorange"),
        "agn_simbad_confirmed": ("o", "steelblue"),
        "agn_simbad_candidate": ("s", "mediumpurple"),
        "agn_tns_flare": ("D", "seagreen"),
        "agn_cats_nonperiodic_hostgalaxy": ("P", "goldenrod"),
        "agn_cats_nonperiodic_pointlike": ("x", "grey"),
    }

    for group, (marker, color) in group_markers.items():
        df_g = df_agn_selected[df_agn_selected["group"] == group]
        if df_g.empty:
            continue
        ax.scatter(
            df_g["ra"].astype(float),
            df_g["dec"].astype(float),
            marker=marker,
            s=45,
            color=color,
            alpha=0.85,
            zorder=3,
            label=f"{group} (N={len(df_g)})",
        )

    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    ax.set_title(f"AGN candidates in LSST Deep Drilling Fields (N={len(df_agn_selected)})")
    ax.legend(fontsize=7, loc="best")
    plt.tight_layout()
    savefig("03_agn_sky_map")
    plt.show()

## 13. Save all light curves to Parquet

`*_src.parquet` files contain all dipole columns (`r:isDipole`, `r:dipoleFluxDiff`,
etc.) in addition to the standard photometric and DRP columns.

In [ ]:
for group in all_groups:
    oids = group_oids[group]
    all_fp, all_src = [], []
    for oid in oids:
        d = lc_cache.get(oid, {})
        for df, store in ((d.get("fp", pd.DataFrame()), all_fp), (d.get("src", pd.DataFrame()), all_src)):
            if not df.empty:
                tmp = df.copy()
                tmp["diaObjectId"] = oid
                tmp["group"] = group
                tmp["field"] = d.get("field")
                tmp["baseline_days"] = d.get("baseline_days")
                store.append(tmp)
    safe = group.replace("/", "_")
    for store, tag in ((all_fp, "fp"), (all_src, "src")):
        if store:
            path = os.path.join(DIR_DATA, f"{safe}_{tag}.parquet")
            pd.concat(store, ignore_index=True).to_parquet(path, index=False)
            print(f"  Saved {path}")

print("\nAll light curve data saved.")

## 14. Final summary

In [ ]:
if df_var.empty:
    print("No variability data.")
else:
    summary = (
        df_var.groupby(["group", "band"])
        .agg(
            n_obj=("diaObjectId", "nunique"),
            n_pts=("n_pts", "sum"),
            median_fvar=("fvar", "median"),
            median_baseline=("baseline_days", "median"),
        )
        .reset_index()
        .sort_values(["band", "median_fvar"], ascending=[True, False])
    )
    print("AGN candidate summary by group and band:")
    print("=" * 90)
    print(summary.to_string(index=False, float_format="{:.3f}".format))

print(f"\nTotal AGN light curves downloaded : {len(lc_cache)}")
print(f"Groups                             : {sorted(set(d['group'] for d in lc_cache.values()))}")
print(f"Fields covered                     : {sorted(set(d.get('field') for d in lc_cache.values()))}")

## 15. Notes and next steps

### Tuning the AGN selection

- `NP_MIN_AGN` and `MIN_BASELINE_DAYS` (section 6) directly trade off **sample size**
  vs. **light-curve quality**. If too few candidates survive, relax `MIN_BASELINE_DAYS`
  first (baseline grows automatically as more LSST DDF seasons are observed).
- The `agn_cats_nonperiodic_pointlike` tier is the least reliable (no galaxy
  counterpart); treat it as a lower-purity sample and cross-check individually
  (e.g. via `r:extendedness`, colour, or a resolver query to SIMBAD/NED) before
  using it for science.
- `CATS_SCORE_MIN` controls the purity/completeness trade-off of the ML-based tiers.
- **Dense-field truncation**: `SATURATION_FRAC` (section 1) controls when a field is
  considered truncated and triggers the tiled fallback search (section 5). Lower it
  if COSMOS-like fields still look under-sampled after the fallback runs.
- **diaObjectId precision**: the `/api/v1/objects` batch enrichment (section 6) uses
  positional ID matching by default, which sidesteps JSON float precision loss on
  large diaObjectId values -- the original cause of an all-NaN `baseline_days` column.

### Possible extensions

- **Damped Random Walk (DRW) fitting**: fit each multi-band light curve with a DRW/
  Ornstein-Uhlenbeck model (e.g. `celerite2`, `taufit`) to extract the characteristic
  damping timescale tau and long-term variance SF_inf -- both correlate with AGN
  black-hole mass and luminosity (MacLeod et al. 2010, Suberlak et al. 2021).
- **Structure function analysis**: compute the first-order structure function
  SF(Delta t) per band and compare its slope to the expected DRW power-law index.
  Needs the long, densely sampled DDF baselines targeted by the `MIN_BASELINE_DAYS` cut.
  A dedicated notebook `02_...` could combine this with `06_FGCM` atmospheric
  parameters to check for residual atmospheric-transparency systematics in the
  AGN structure function at short lags.
- **Cross-match with external AGN catalogues** (Milliquas, SDSS quasar catalogue,
  Gaia AGN candidates) via `/api/v1/resolver` to validate/expand the
  `agn_cats_nonperiodic_*` tiers.
- **Colour-colour and colour-magnitude diagrams** using the per-band
  `scienceFluxMean` from `/api/v1/objects` to further separate AGN from
  contaminants (blue power-law continuum vs. stellar colours).